# Metricas para resultados de `evaluate_system.py`

Notebook minimo para Colab: carga el CSV generado por `evaluate_system.py`, usa solo la columna `final_prediction` y calcula las mismas metricas de `pipeline_v3.ipynb` contra `ground_truth`.

In [ ]:
!pip install -q rouge-score==0.1.2 bert-score==0.3.13 transformers==4.46.3 pandas numpy

In [ ]:
import os
import json
from datetime import datetime

import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Mismas carpetas del pipeline_v3.ipynb
DRIVE_BASE = '/content/drive/MyDrive/dataset eval'
DRIVE_RESULTS_FOLDER = DRIVE_BASE + '/resultados/base+rag'
os.makedirs(DRIVE_RESULTS_FOLDER, exist_ok=True)

RESULT_CSV_PATH = DRIVE_BASE + '/results_NetGen_system_RAG_Batfish_20260622_210434.csv'

print('Carpeta resultados: ' + DRIVE_RESULTS_FOLDER)

In [ ]:
if not os.path.exists(RESULT_CSV_PATH):
    raise FileNotFoundError('No encontre el CSV: ' + RESULT_CSV_PATH)

df = pd.read_csv(RESULT_CSV_PATH, encoding='utf-8-sig')

required_cols = ['final_prediction', 'ground_truth']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError('Faltan columnas en el CSV: ' + ', '.join(missing))

predictions = df['final_prediction'].fillna('').tolist()
references = df['ground_truth'].fillna('').tolist()

print('CSV cargado: ' + RESULT_CSV_PATH)
print('Muestras  : ' + str(len(df)))

In [ ]:
def normalize_config(text):
    '''Elimina prompts CLI, configure terminal y end para comparacion justa.
    El prompt v3 genera configure terminal/end; se eliminan aqui para que
    el ROUGE sea comparable con el ground truth de topo_v1 que no los tiene.
    '''
    lines = []
    for line in text.split(chr(10)):
        if '#' in line:
            line = line.split('#', 1)[-1]
        elif '>' in line:
            line = line.split('>', 1)[-1]
        line = ' '.join(line.split()).lower()
        if line and line not in ('configure terminal', 'end', 'enable', 'no_code'):
            lines.append(line)
    return lines


def compute_rouge(predictions, references):
    from rouge_score import rouge_scorer as rs
    scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred or not ref or pred == 'ERROR':
            continue
        pred_norm = ' '.join(normalize_config(pred))
        ref_norm  = ' '.join(normalize_config(ref))
        if not pred_norm or not ref_norm:
            continue
        s = scorer.score(ref_norm, pred_norm)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s['rougeL'].fmeasure)

    def safe(lst):    return round(float(np.mean(lst)), 4) if lst else 0.0
    def safestd(lst): return round(float(np.std(lst)),  4) if lst else 0.0

    return {
        'rouge1': safe(r1), 'rouge1_std': safestd(r1),
        'rouge2': safe(r2), 'rouge2_std': safestd(r2),
        'rougeL': safe(rl), 'rougeL_std': safestd(rl),
    }


def compute_bertscore(predictions, references):
    from bert_score import score as bscore
    valid = [(p, r) for p, r in zip(predictions, references)
             if p and r and p != 'ERROR']
    if not valid:
        return {'bertscore_p': 0.0, 'bertscore_r': 0.0,
                'bertscore_f1': 0.0, 'bertscore_f1_std': 0.0}
    preds, refs = zip(*valid)
    for model_name, num_layers in [('microsoft/codebert-base', 12), ('roberta-large', None)]:
        try:
            kwargs = {'lang': 'en', 'model_type': model_name,
                      'verbose': False, 'batch_size': 16}
            if num_layers:
                kwargs['num_layers'] = num_layers
            P, R, F1 = bscore(list(preds), list(refs), **kwargs)
            print('    BERTScore model: ' + model_name)
            return {
                'bertscore_p':      round(float(P.mean()),  4),
                'bertscore_r':      round(float(R.mean()),  4),
                'bertscore_f1':     round(float(F1.mean()), 4),
                'bertscore_f1_std': round(float(F1.std()),  4),
            }
        except Exception as e:
            print('    ' + model_name + ': ' + str(e))
    return {'bertscore_p': 0.0, 'bertscore_r': 0.0,
            'bertscore_f1': 0.0, 'bertscore_f1_std': 0.0}


print('Metricas definidas')

In [ ]:
print('Calculando ROUGE (normalizado)...')
rouge_metrics = compute_rouge(predictions, references)

print('Calculando BERTScore...')
bert_metrics = compute_bertscore(predictions, references)

metrics = {**rouge_metrics, **bert_metrics}

print('ROUGE-1      : {:.4f} (+-{:.4f})'.format(metrics['rouge1'], metrics['rouge1_std']))
print('ROUGE-2      : {:.4f} (+-{:.4f})'.format(metrics['rouge2'], metrics['rouge2_std']))
print('ROUGE-L      : {:.4f} (+-{:.4f})'.format(metrics['rougeL'], metrics['rougeL_std']))
print('BERTScore-F1 : {:.4f} (+-{:.4f})'.format(metrics['bertscore_f1'], metrics['bertscore_f1_std']))

In [ ]:
output_path = os.path.join(
    DRIVE_RESULTS_FOLDER,
    'metrics_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.json'
)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print('OK metricas guardadas:')
print('   ' + output_path)